# MultiAgentic RAG Using Langchain #

In [5]:
import warnings 
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader 

loader = PyPDFLoader('economics_research_reference.pdf')
pages = loader.load()


In [ ]:
# Building chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=140)
texts=text_spliter.split_documents(pages)
chunks = [i.page_content for i in texts]
metadata = [i.metadata for i in texts]

print(f"Embedding {len(chunks)} chunks...")


Embedding 31 chunks...


: 

: 

: 

: 

: 

: 

In [ ]:
# Create VectorDB 
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
embedding_function = DefaultEmbeddingFunction()

client = chromadb.PersistentClient(path="./Database_VD")
collection = client.get_or_create_collection(name="clean_vectorDB",embedding_function=embedding_function)

if collection.count()==0:
    collection.add(
        documents=chunks,
        ids=[str(i) for i in range(len(chunks))],
        metadatas=metadata
    )


: 

: 

: 

: 

: 

: 

In [ ]:
# Import models and local enviroment 
from langchain_groq import ChatGroq 
from dotenv import load_dotenv 

# Import tools 
from langchain_core.tools import tool 
from langchain_community.tools import DuckDuckGoSearchRun  

load_dotenv()
import os 

key = os.getenv('GROQ_API_KEY')
chat_model = ChatGroq(model='llama-3.1-8b-instant',api_key=key) 


: 

: 

: 

: 

: 

: 

In [ ]:
@tool 
def calculator(Exception:str):
    """ do the calculations based on the user given expression """
    return str(eval(Exception))


@tool 
def retrival(query:str):
    """ provide the answer of user's qustions based on the given local document  """
    result = collection.query(query_texts=[query],n_results=5)
    distance = result['distances'][0]
    document = result['documents'][0]
    thresold = 1.0 

    near_docs = []
    for i , doc in zip(distance,document):
        if i < thresold :
            near_docs.append(doc)

    if not near_docs:
        return "NOT relevent Content"
    return "\n\n".join(near_docs)

@tool
def web_search(query:str):
    """ use this tool when the answer or relavent docs can not find out on the given local document """
    search = DuckDuckGoSearchRun()
    return search.run(query)


: 

: 

: 

: 

: 

: 

In [ ]:
combine_tools=[calculator,retrival,web_search]
tool_map={t.name:t for t in combine_tools} 
tool_map


{'calculator': StructuredTool(name='calculator', description='do the calculations based on the user given expression', args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x11a4f9280>),
 'retrival': StructuredTool(name='retrival', description="provide the answer of user's qustions based on the given local document", args_schema=<class 'langchain_core.utils.pydantic.retrival'>, func=<function retrival at 0x11a4f9550>),
 'web_search': StructuredTool(name='web_search', description='use this tool when the answer or relavent docs can not find out on the given local document', args_schema=<class 'langchain_core.utils.pydantic.web_search'>, func=<function web_search at 0x11a4f9af0>)}

: 

: 

: 

: 

: 

: 

In [ ]:
calculator_agent = chat_model.bind_tools([calculator])
retrival_agent = chat_model.bind_tools([retrival])
web_agent = chat_model.bind_tools([web_search])

AGENTS={
    "cal":calculator_agent,
    "retrive":retrival_agent,
    "web":web_agent
}
# Sucessfully create agents 
AGENTS


{'cal': RunnableBinding(bound=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x116149040>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11902c8e0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'calculator', 'description': 'do the calculations based on the user given expression', 'parameters': {'properties': {'Exception': {'type': 'string'}}, 'required': ['Exception'], 'type': 'object'}}}]}, config={}, config_factories=[]),
 'retrive': RunnableBinding(bound=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x116149040>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11902c8e0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'fu

: 

: 

: 

: 

: 

: 

In [ ]:
def fetch(query:str):
    prompt = f"""
    Always define user asking qustions for one tool . calculator , retrive , web_search,

    before provide answers follow the instructions :
    1. any calculation or operations use calculator . example : 23*78 only arithmetic operation either everything transfer to the retrive tool
    2. after all for every qustions go for retrive if their cant find any contextual relations then go to the web_search tool , but before that dont go the web_search tool.

    qustion : {query}
     """
    response = chat_model.invoke(prompt)
    print(response.content)
    label = response.content.strip().lower()
    for key in AGENTS:
        if key in label:
            return key 
    return 'General'


: 

: 

: 

: 

: 

: 

In [ ]:
def run (query:str , state:dict=None):
    state = state or {"history":[]}

    label = fetch(query)

    if label=="General":
        answer = chat_model.invoke(query)
    else :
        agent_select=AGENTS[label]
        response = agent_select.invoke(query)

        if not response.tool_calls:
            answer = response.content 
        else:
            call = response.tool_calls[0]
            tool_name = tool_map[call['name']]
            answer=tool_name.invoke(call['args']) 
    
    state['history'].append(
        {
            "query": query , 
            "state": state,
            "answer" : answer
        }
    )

    return answer , state  


: 

: 

: 

: 

: 

: 

In [ ]:
state = None 

answer_1 , state = run (' america president ?' ,state)

print(answer_1)


To answer your question "america president", I'll first try to retrieve the information from our knowledge database.

However, the exact question is very general and can have multiple answers. To narrow it down, I'll consider the current president of the United States.

Retrieving information from our database...

Unfortunately, I couldn't find any specific information about the current president of the United States in our database.

Since I couldn't find the information in our database, I'll move on to the web search tool to find the answer for you.

(If you want to know the current president, I can suggest searching for "current president of the United States" or "USA president" on a search engine.)
The incumbent president is Donald Trump, who assumed office on January 20, 2025. [5][6] Since the office was established in 1789, 45 men have served in 47 presidencies. The president of the United States (POTUS) [b] is the head of state and head of government of the United States. The pr

: 

: 

: 

: 

: 

: 

In [ ]:
query = "what is 78*78 "
label = fetch(query)
print(f"[routed to: {label}]")   


**User Asking Question:** To calculate the product of 78 and 78 using the calculator.

**Answer:** 

Using the calculator tool, I will perform the operation: 
78 * 78 = 

Calculating... 
78 * 70 = 5460
78 * 8 = 624
Adding the results... 
5460 + 624 = 6084

**Answer:** The product of 78 and 78 is 6084.
[routed to: cal]


: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 